In [12]:
import numpy as np
import pandas as pd
from scipy import stats

In [13]:
df = pd.read_csv("../data/MachineLearningRating_v3.txt", sep="|")
df.head()

C:\Users\nbe\AppData\Local\Temp\ipykernel_5020\455118313.py:1: DtypeWarning: Columns (0: CapitalOutstanding, 1: CrossBorder) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/MachineLearningRating_v3.txt", sep="|")


,UnderwrittenCoverID,PolicyID,TransactionMonth,IsVATRegistered,Citizenship,LegalType,Title,Language,Bank,AccountType,...,ExcessSelected,CoverCategory,CoverType,CoverGroup,Section,Product,StatutoryClass,StatutoryRiskType,TotalPremium,TotalClaims
0,145249,12827,2015-03-01 00:00:00,True,,Close Corporation,Mr,English,First National Bank,Current account,...,Mobility - Windscreen,Windscreen,Windscreen,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,21.929825,0.0
1,145249,12827,2015-05-01 00:00:00,True,,Close Corporation,Mr,English,First National Bank,Current account,...,Mobility - Windscreen,Windscreen,Windscreen,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,21.929825,0.0
2,145249,12827,2015-07-01 00:00:00,True,,Close Corporation,Mr,English,First National Bank,Current account,...,Mobility - Windscreen,Windscreen,Windscreen,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,0.000000,0.0
3,145255,12827,2015-05-01 00:00:00,True,,Close Corporation,Mr,English,First National Bank,Current account,...,Mobility - Metered Taxis - R2000,Own damage,Own Damage,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,512.848070,0.0
4,145255,12827,2015-07-01 00:00:00,True,,Close Corporation,Mr,English,First National Bank,Current account,...,Mobility - Metered Taxis - R2000,Own damage,Own Damage,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,0.000000,0.0


In [15]:
df.replace([np.inf, -np.inf], np.nan, inplace=True)

df["TotalClaims"] = df["TotalClaims"].fillna(0)
df["TotalPremium"] = df["TotalPremium"].fillna(df["TotalPremium"].median())

In [16]:
def claim_frequency(data):
    return (data["TotalClaims"] > 0).mean()

In [17]:
def claim_severity(data):
    claims = data[data["TotalClaims"] > 0]["TotalClaims"]
    return claims.mean()

In [18]:
def margin(data):
    return data["TotalPremium"] - data["TotalClaims"]

In [19]:
gauteng = df[df["Province"] == "Gauteng"]
wc = df[df["Province"] == "Western Cape"]

In [20]:
t_stat, p_val = stats.ttest_ind(
    gauteng["TotalClaims"],
    wc["TotalClaims"],
    nan_policy="omit"
)

print("p-value:", p_val)

p-value: 0.05632044649871941


In [21]:
alpha = 0.05

if p_val < alpha:
    print("Reject H0 - Significant difference")
else:
    print("Fail to Reject H0 - No evidence of difference")

Fail to Reject H0 - No evidence of difference


In [22]:
results = []

In [23]:
results.append({
    "Hypothesis": "Province (Gauteng vs Western Cape)",
    "Test": "t-test",
    "p-value": p_val,
    "Decision": "Reject H0" if p_val < 0.05 else "Fail to Reject H0"
})

In [24]:
pd.DataFrame(results)

,Hypothesis,Test,p-value,Decision
0,Province (Gauteng vs Western Cape),t-test,0.05632,Fail to Reject H0


In [25]:
male = df[df["Gender"] == "Male"]
female = df[df["Gender"] == "Female"]

In [26]:
male_freq = (male["TotalClaims"] > 0).mean()
female_freq = (female["TotalClaims"] > 0).mean()

print("Male frequency:", male_freq)
print("Female frequency:", female_freq)

Male frequency: 0.0021953896816684962
Female frequency: 0.002072538860103627


In [27]:
contingency = pd.crosstab(
    df["Gender"],
    df["TotalClaims"] > 0
)
contingency

TotalClaims,False,True
Gender,,
Female,6741,14
Male,42723,94
Not specified,938324,2666


In [28]:
chi2, p_val, dof, expected = stats.chi2_contingency(contingency)

print("p-value:", p_val)

p-value: 0.026570248768437145


In [29]:
alpha = 0.05

if p_val < alpha:
    print("Reject H0 - Gender affects claim frequency")
else:
    print("Fail to Reject H0 - No evidence of difference")

Reject H0 - Gender affects claim frequency


In [30]:
results.append({
    "Hypothesis": "Gender vs Claim Frequency",
    "Test": "Chi-square",
    "p-value": p_val,
    "Decision": "Reject H0" if p_val < 0.05 else "Fail to Reject H0"
})

In [31]:
pd.DataFrame(results)

,Hypothesis,Test,p-value,Decision
0,Province (Gauteng vs Western Cape),t-test,0.05632,Fail to Reject H0
1,Gender vs Claim Frequency,Chi-square,0.02657,Reject H0


In [32]:
df_claims = df[df["TotalClaims"] > 0]

In [33]:
df_claims["VehicleType"].value_counts()

VehicleType
Passenger Vehicle    2587
Medium Commercial     158
Heavy Commercial       21
Light Commercial        8
Bus                     1
Name: count, dtype: int64

In [34]:
bus = df_claims[df_claims["VehicleType"] == "Bus"]["TotalClaims"]
passenger = df_claims[df_claims["VehicleType"] == "Passenger Vehicle"]["TotalClaims"]

In [35]:
t_stat, p_val = stats.ttest_ind(
    bus,
    passenger,
    nan_policy="omit"
)

print("p-value:", p_val)

p-value: 0.6790323656388665


In [36]:
alpha = 0.05

if p_val < alpha:
    print("Reject H0 - Vehicle type affects claim severity")
else:
    print("Fail to Reject H0 - No evidence of difference")

Fail to Reject H0 - No evidence of difference


In [37]:
results.append({
    "Hypothesis": "Vehicle Type vs Claim Severity",
    "Test": "t-test",
    "p-value": p_val,
    "Decision": "Reject H0" if p_val < 0.05 else "Fail to Reject H0"
})

In [38]:
pd.DataFrame(results)

,Hypothesis,Test,p-value,Decision
0,Province (Gauteng vs Western Cape),t-test,0.056320,Fail to Reject H0
1,Gender vs Claim Frequency,Chi-square,0.026570,Reject H0
2,Vehicle Type vs Claim Severity,t-test,0.679032,Fail to Reject H0


In [39]:
df["Margin"] = df["TotalPremium"] - df["TotalClaims"]

In [40]:
df["PostalCode"].value_counts().head()

PostalCode
2000    133498
122      49171
7784     28585
299      25546
7405     18518
Name: count, dtype: int64

In [41]:
top_zips = df["PostalCode"].value_counts().head(2).index

zip1 = df[df["PostalCode"] == top_zips[0]]["Margin"]
zip2 = df[df["PostalCode"] == top_zips[1]]["Margin"]

In [42]:
t_stat, p_val = stats.ttest_ind(
    zip1,
    zip2,
    nan_policy="omit"
)

print("p-value:", p_val)

p-value: 0.19590898368555731


In [43]:
alpha = 0.05

if p_val < alpha:
    print("Reject H0 - Zip codes differ in profitability")
else:
    print("Fail to Reject H0 - No evidence of difference")

Fail to Reject H0 - No evidence of difference


In [44]:
results.append({
    "Hypothesis": "Zip Code vs Margin",
    "Test": "t-test",
    "p-value": p_val,
    "Decision": "Reject H0" if p_val < 0.05 else "Fail to Reject H0"
})

In [45]:
final_results = pd.DataFrame(results)
final_results

,Hypothesis,Test,p-value,Decision
0,Province (Gauteng vs Western Cape),t-test,0.056320,Fail to Reject H0
1,Gender vs Claim Frequency,Chi-square,0.026570,Reject H0
2,Vehicle Type vs Claim Severity,t-test,0.679032,Fail to Reject H0
3,Zip Code vs Margin,t-test,0.195909,Fail to Reject H0



## BUSINESS INTERPRETATION:

1. Provinces:
No strong statistical evidence of claim differences between provinces. Pricing can remain largely consistent across regions, with continued monitoring for emerging regional risk shifts.

2. Gender:
A significant difference in claim frequency suggests gender is a meaningful risk factor. This may justify more refined segmentation in pricing or underwriting strategies.

3. Vehicle Type:
No significant difference in claim severity across vehicle types in this grouping. Vehicle type alone is not a strong driver of claim cost in this dataset.

4. Zip Code:
No significant difference in profitability across zip codes. Current pricing appears geographically stable at this level of granularity.
"""